In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt

from utils.geo import haversine_distance

#seed = 14
np.random.seed(random.randrange(1, 100))

n_nodes = 10
k=3
n_simulations = 1
alpha_value = 0.1
alphazero = 1

cell_dataset = pd.read_parquet("assets/porto_cells.parquet")
output = f"TON_IoT_{n_nodes}n_{k}k"
bbox_img = "assets/background.png"
bbox_boundaries = "assets/BBox_Porto.txt"

In [ ]:
with open(bbox_boundaries, "r") as f:
    box = eval(f.readline())
    
img = plt.imread(bbox_img)

def plot_network(network: nx.Graph, path: Path, towers: np.ndarray) -> None:
    fig, ax = plt.subplots(figsize=(6,6))    
    
    ax.set_xlim([box[0], box[1]])
    ax.set_ylim([box[2], box[3]])
    
    aspect_ratio = abs((box[1]-box[0])/(box[2]-box[3]))

    ax.set_aspect(aspect_ratio)

    ax.imshow(img, extent=box, aspect=aspect_ratio)
        
    nx.draw_networkx(
            network,
            pos=towers,
            with_labels=True,
            node_color='gold',
            node_size=150,
            width=1.5,
            ax=ax,
            font_size=8,
            edgecolors="black",
        )
    ax.set_axis_on()

    plt.savefig(path, dpi=500)

def build_network(
    cell_dataset: pd.DataFrame, num_towers: int, k_edge_connectivity: int
) -> tuple[nx.Graph, pd.DataFrame]:
    """
    Builds a network of towers.

    :param cell_dataset: the dataset containing tower positions
    :param num_towers: the number of towers (nodes) to use for the network
    :param k_edge_connectivity: k edge connectivity of the desired graph
    :return: a tuple containing as first element the networkx graph object, as second element a
    dataframe with node coordinates
    """
    while True:  # when a result is found, it is directly returned
        # sample the required number of towers
        towers = cell_dataset.sample(num_towers, replace=False)
        towers.reset_index(drop=True, inplace=True)

        # calculate tuples of node distances (v, w, dist(v,w))
        distances = set()
        for i in range(num_towers):
            for j in range(i + 1, num_towers):
                dist = haversine_distance(
                    longitude_1=towers["lon"].astype(float)[i],
                    longitude_2=towers["lon"].astype(float)[j],
                    latitude_1=towers["lat"].astype(float)[i],
                    latitude_2=towers["lat"].astype(float)[j],
                )
                distances.add((i, j, dist))

        network = nx.Graph()
        network.add_nodes_from(range(num_towers))

        edges = nx.k_edge_augmentation(network, k_edge_connectivity, avail=distances)
        network.add_edges_from(edges)

        return network, towers[["lon", "lat"]]

In [ ]:
for i in range(n_simulations):
    folder_path = Path(f"data/networks/{output}/{i}")
    folder_path.mkdir(parents=True, exist_ok=True)
    folder_path2 = Path(f"data/networks/{output}/{i+1}")
    folder_path2.mkdir(parents=True, exist_ok=True)

    network, towers = build_network(cell_dataset, num_towers=n_nodes, k_edge_connectivity=k)

    plot_network(network, path=folder_path / "plot.jpg", towers=towers.values)
    nx.write_adjlist(network, folder_path / "adj_list.txt")
    towers.to_csv(folder_path / "nodes.csv", index=False)

    plot_network(network, path=folder_path2 / "plot.jpg", towers=towers.values)
    nx.write_adjlist(network, folder_path2 / "adj_list.txt")
    towers.to_csv(folder_path2 / "nodes.csv", index=False)

    print("Saved!")

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder


# Caricamento dataset
def f(i):
    return pd.read_csv(i, 
                       usecols=["ts","src_bytes","missed_bytes","label","type","conn_state", "duration"], 
                       na_values="-"
    )

df = pd.concat(map(f, [ 
                       'assets/Processed_Network_dataset/Network_dataset_2.csv',
                       'assets/Processed_Network_dataset/Network_dataset_3.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_4.csv',
                       'assets/Processed_Network_dataset/Network_dataset_5.csv',
                       'assets/Processed_Network_dataset/Network_dataset_6.csv',
                       'assets/Processed_Network_dataset/Network_dataset_7.csv',
                       'assets/Processed_Network_dataset/Network_dataset_8.csv',
                       'assets/Processed_Network_dataset/Network_dataset_9.csv',
                       'assets/Processed_Network_dataset/Network_dataset_10.csv',
                       'assets/Processed_Network_dataset/Network_dataset_11.csv',
                       'assets/Processed_Network_dataset/Network_dataset_12.csv',
                       'assets/Processed_Network_dataset/Network_dataset_13.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_14.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_15.csv',
                       'assets/Processed_Network_dataset/Network_dataset_16.csv', 
                       'assets/Processed_Network_dataset/Network_dataset_17.csv',
                       'assets/Processed_Network_dataset/Network_dataset_18.csv',
                       'assets/Processed_Network_dataset/Network_dataset_19.csv',
                       'assets/Processed_Network_dataset/Network_dataset_20.csv',
                       'assets/Processed_Network_dataset/Network_dataset_21.csv']))
df = df.dropna()
print("Count: ", df['type'].value_counts())


In [ ]:
target_col = df.columns[-1]
y = df[target_col]
df = df.drop(columns=[target_col])

#separazione colonne
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
boolean_cols = [col for col in df.columns if df[col].dropna().nunique() == 2 and df[col].dropna().isin([0, 1]).all()]
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.difference(boolean_cols).tolist()

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

scaler = StandardScaler()
cols_to_scale = list(set(numeric_cols) & set(df_encoded.columns))
df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])
x = df_encoded.astype('float32').values
print(x.shape) 

le = LabelEncoder()
y_encoded = le.fit_transform(y)  # ['normal', 'dos', ...] → [0, 1, ...]

In [ ]:
#Dataset creation
import numpy as np
from utils.data import train_val_test_split

def dirichlet_split(y, n_clients, alpha):
    num_c = len(np.unique(y))
    net_dataidx_map = {}
    p_client = np.zeros((n_clients, num_c))
    for i in range(n_clients):
        p_client[i] = np.random.dirichlet(np.repeat(alpha, num_c))
    idx_batch = [[] for _ in range(n_clients)]
    for k in range(num_c):
        idx_k = np.where(y == k)[0]
        np.random.shuffle(idx_k)
        proportions = p_client[:, k]
        proportions = proportions / proportions.sum()
        split_points = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]
        idx_batch = [
            idx_j + idx.tolist()
            for idx_j, idx in zip(idx_batch, np.split(idx_k, split_points))
        ]
    for j in range(n_clients):
        np.random.shuffle(idx_batch[j])
        net_dataidx_map[j] = idx_batch[j]
    net_cls_counts = {}
    for net_i, dataidx in net_dataidx_map.items():
        unq, unq_cnt = np.unique(y.iloc[dataidx], return_counts=True)
        net_cls_counts[net_i] = dict(zip(unq, unq_cnt))
    return net_dataidx_map, net_cls_counts


#Y_train_prova = next(iter(train_loader))[1].numpy()
#inds, net_cls_counts = dirichlet_split(Y_train_prova, 10, 0.05, 10)

#print("Inds", inds)
#print("Net cls count", net_cls_counts)

#n_nodes = 10
#k=3
#n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

df_full = df_encoded.copy()
df_full[target_col] = y_encoded
df_full = df_full.sample(frac=1, random_state=np.random.RandomState()).reset_index(drop=True)
#splits = np.array_split(df_full, n_nodes)
#splits = np.array_split(df_full, 10)

#for i in range(n_simulations):
i = 0
split_df = df_full
dataset_folder = Path(output_path / str(i) / f"4inTestBalancedAlpha0{alphazero}_seedRand")
dataset_folder.mkdir(parents=True, exist_ok=True)
dataset_folder2 = Path(output_path / str(i+1) / f"4inTestBalancedAlpha0{alphazero}_seedRand")
dataset_folder2.mkdir(parents=True, exist_ok=True)

target_col = split_df.columns[-1]
y = split_df[target_col]
x = split_df.drop(columns=[target_col])

node_map, _ = dirichlet_split(y, n_clients=n_nodes, alpha=alpha_value)

for node_id, idxs in node_map.items():
    X_node = x.iloc[idxs]
    Y_node = y.iloc[idxs]

    X_train, Y_train, X_val, Y_val, X_test, Y_test = train_val_test_split(
            X_node, Y_node, test_perc=0.3, val_perc_on_train=0.1
        )

    np.savez(
            dataset_folder / f"node_{node_id}.npz",
            X_train=X_train.to_numpy(),
            Y_train=Y_train.to_numpy(),
            X_val=X_val.to_numpy(),
            Y_val=Y_val.to_numpy(),
            X_test=X_test.to_numpy(),
            Y_test=Y_test.to_numpy(),
        )
    
    np.savez(
            dataset_folder2 / f"node_{node_id}.npz",
            X_train=X_train.to_numpy(),
            Y_train=Y_train.to_numpy(),
            X_val=X_val.to_numpy(),
            Y_val=Y_val.to_numpy(),
            X_test=X_test.to_numpy(),
            Y_test=Y_test.to_numpy(),
        )
    #split_df.to_parquet(dataset_folder / f"node_{node_id}.parquet", index=False)
    print(f"Node {node_id} in simulation {i} saved!")
print(f"Simulation {i} saved!")


In [ ]:
#Reset del Kernel
%reset -s -f

#import os
#os._exit(00)


#from IPython.core.display import HTML
#HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import functools
import json
import os
from pathlib import Path
import torch
import torch.nn as nn

from gossiplearning.config import Config
from utils.gossip_training import get_node_dataset, round_trip_fn, model_transmission_fn, \
    run_simulation
from utils.model_creators import create_MLP
from utils.multiprocessing_test import run_in_parallel

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf

n_nodes = 10
k=3
n_simulations = 10
alphazero = 1
#n_nodes = 2
#k=1
#n_simulations = 1
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")
output_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")

In [ ]:
import functools
import os
import json
from pathlib import Path
import torch
import torch.nn as nn

from gossiplearning.config import Config
from utils.model_creators import create_MLP
from utils.single_node_training import train_single_nodes

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

simulation_number = 0

#timesteps = 4
dataset_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")
with open("TON_IoT_config.json", "r") as f:
    config = json.load(f)

config = Config.model_validate(config)

dataset_path = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k")
network_path = Path(f"data/networks/TON_IoT_{n_nodes}n_{k}k")

In [ ]:
i = 0
train_single_nodes(
     datasets_folder=dataset_path / str(simulation_number) / f"4inTestBalancedAlpha0{alphazero}_seedRand",
     output_folder=dataset_path / str(simulation_number) / f"4inTestBalancedAlpha0{alphazero}models" / "models",
     config=config,
     model_creator=functools.partial(create_MLP, config=config),
)

In [ ]:
#Reset del Kernel
%reset -s -f

In [ ]:
from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch
from utils.data import load_npz_data
import numpy as np

simulation_number = 0
n_nodes = 10
k = 3
alphazero = 1

#4inTestBalanced
model = tuple([torch.load(f"data/datasets/TON_IoT_{n_nodes}n_{k}k/{simulation_number}/4inTestBalancedAlpha0{alphazero}models/models/node_{i}/single_node_{i}.pt") for i in range(n_nodes)])
datasets = tuple([load_npz_data(f"data/datasets/TON_IoT_{n_nodes}n_{k}k/{simulation_number}/4inTestBalancedAlpha0{alphazero}_seedRand/node_{i}.npz") for i in range(n_nodes)])

X = np.concatenate([d[4] for d in datasets])
Y = np.concatenate([d[5] for d in datasets])
L = np.concatenate([d[1] for d in datasets])
tot_len = len(L)

n_samples = 3500000
idx = np.random.choice(len(X), size=n_samples, replace=False)

In [ ]:
node = 0
f1_weights = []
for d in datasets:
    node_weight = len(d[1])/tot_len
    print(f"Node{node} weight: ", node_weight)
    node = node+1
    f1_weights.append(node_weight)

print("Sum of all weights: ", sum(f1_weights))

In [ ]:
from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from utils.metrics import compute_metrics

import seaborn as sns

class MLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        input_dim = 17
        self.fc1 = nn.Linear(input_dim, 128)
        #self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        #self.dropout2 = nn.Dropout(0.3)
        self.output_layer = nn.Linear(64, 7)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        #x = self.dropout2(x)
        logit = self.output_layer(x)  
        return logit


x_tensor = torch.tensor(X[idx].astype("float32"), dtype=torch.float32)
y_tensor = torch.tensor(Y[idx], dtype=torch.long)
num_classes = 7
#num_nodes = 20
error_matrix = np.zeros((n_nodes, num_classes))

print("X: ", x_tensor.shape)
print("Y: ", y_tensor.shape)
accuracy_list = []
history = []

for i in range(n_nodes):
    print("Initializing model from node ", i)
    loaded_model = MLPModel()
    loaded_model.load_state_dict(model[i])

    loaded_model.eval()
    with torch.no_grad():
        logits = loaded_model(x_tensor)              
        y_pred = torch.argmax(logits, dim=1)         
        correct = (y_pred == y_tensor).sum().item()
        accuracy_ = correct / len(y_tensor)

    print("Node: ", str(i))
    print("Accuracy:", str(accuracy_), "\n")
    accuracy_list.append(accuracy_)
    probs = torch.softmax(logits, dim=1)

    predictions_gos = y_pred.numpy()
    true_pred = y_tensor.numpy()
    metrics = compute_metrics(true_pred, predictions_gos)
    dict_history = {
        "Node": str(i),
        "accuracy": metrics.acc,
        "precision_macro": metrics.prec,
        "recall_macro": metrics.rec,
        "f1_macro": metrics.f1,
        "f1_weighted": metrics.f1_weighted,
        "weight": f1_weights[i]
    }
    history.append(dict_history)
    target_names = [str(c) for c in np.unique(true_pred)]
    report = classification_report(true_pred, 
                                        predictions_gos, 
                                        labels=[0,1,2,3,4,5,6], 
                                        target_names=target_names, zero_division=0,
                                        output_dict=True
                                        )

    cm = confusion_matrix(true_pred, predictions_gos, labels=range(7))
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                        xticklabels=range(7), yticklabels=range(7))
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    path_folder = Path(f"data/datasets/TON_IoT_{n_nodes}n_{k}k/{simulation_number}/4inTestBalancedAlpha0{alphazero}models/testing_unified")
    path_folder.mkdir(parents=True, exist_ok=True)
    plt.savefig(path_folder/ f"matrix_node{i}.png")
    plt.close()

    file_path = path_folder / "report_accuracy.json"
    if not file_path.exists():
        with open(file_path, "w") as outfile:
            json.dump(dict_history, outfile, indent=1)
    else:
        with open(file_path, "r+") as outfile:
            file_data = json.load(outfile)
            if not isinstance(file_data, list):
                file_data = [file_data]
            file_data.append(dict_history)
            outfile.seek(0)
            json.dump(file_data, outfile, indent=1)
        outfile.close
    with open(path_folder/ f"report_node_{i}.json", "w") as outfiler:
                json.dump(report, outfiler, indent=3)
    outfiler.close
    for c in range(num_classes):
        mask = (true_pred == c)
        errors = np.sum(predictions_gos[mask] != c)
        total = np.sum(mask)
        error_matrix[i, c] = errors / total if total > 0 else 0.0

# Plot
x = np.arange(n_nodes)  # nodi
width = 0.1               # larghezza delle barre
plt.figure(figsize=(25,10))

for c in range(num_classes):
    plt.bar(x + c*width, error_matrix[:, c], width, label=f"Classe {c}")

plt.xlabel("Nodo")
plt.ylabel("Errore (%)")
plt.title(f"Percentuale di errore con α = 0.{alphazero}")
plt.xticks(x + (num_classes/2 - 0.5)*width, [f"Nodo {i}" for i in range(n_nodes)])
plt.legend()
plt.tight_layout()
plt.savefig(path_folder/ f"istogramma_err.png")
plt.show()
plt.close()

In [ ]:
import matplotlib.pyplot as plt

xs = [x for x in range(len(history))]

all_acc, all_prec, all_rec, all_f1, all_f1_w = [], [], [], [], []
for i in range(len(history)):
    all_acc.append(history[i].get('accuracy'))
    all_prec.append(history[i].get('precision_macro'))
    all_rec.append(history[i].get('recall_macro'))
    all_f1.append(history[i].get('f1_macro'))
    all_f1_w.append(history[i].get('f1_weighted'))

plt.figure(figsize=(15,12))
plt.xlabel("Nodi")
plt.ylabel("Value %")
plt.xticks(xs)
#plt.plot(xs, accuracy_gossip_list)
plt.plot(
    all_acc,
    label="Gossip Accuracy",
    color="royalblue",
    linewidth=2,
)
plt.plot(
    all_prec,
    label="Gossip Precision",
    color="limegreen",
    linewidth=2,
)
plt.plot(
    all_rec,
    label="Gossip Recall",
    color="darkviolet",
    linewidth=2,
)
plt.plot(
    all_f1,
    label="Gossip F1",
    color="crimson",
    linewidth=2,
)
plt.plot(
    all_f1_w,
    label="Gossip F1",
    color="black",
    linewidth=2,
)

plt.legend()
plt.savefig(Path(path_folder) / "comparison_plot.jpg")
plt.show()
plt.close()

In [ ]:
import statistics
all_value_means = []
#alphas = [0.05, 0.1, 0.3, 0.5, 0.7]
alphas = [3,5,7,9]

all_value_means.append(statistics.mean(all_acc))
all_value_means.append(statistics.mean(all_prec))
all_value_means.append(statistics.mean(all_rec))
all_value_means.append(statistics.mean(all_f1))
all_value_means.append(statistics.mean(all_f1_w))
all_value_means.append(np.average(all_f1_w, weights=f1_weights))


print(all_value_means)

In [ ]:
metrics_history = {
        "N_nodes": n_nodes,
        "Connectivity" : k,
        "Strategy": "Local Training",
        "accuracy": all_value_means[0],
        "precision_macro": all_value_means[1],
        "recall_macro": all_value_means[2],
        "f1_macro": all_value_means[3],
        "f1_weighted": all_value_means[4],
        "f1_weighted_with_nodes_weight": all_value_means[5]
    }

file_path = path_folder / "report_avg.json"
if not file_path.exists():
    with open(file_path, "w") as outfile:
         json.dump(metrics_history, outfile, indent=1)
    outfile.close

In [ ]:
common_met_txt = open(Path(f"txts_comparison/f1_loc_alpha0{alphazero}.txt"), "a")
common_met_txt.write(str(all_value_means[5]))
common_met_txt.write("\n")
common_met_txt.close()

In [ ]:
# Appending
"""file1 = open(Path(f"Aggregators/{aggr}") / f"metrics_with_{aggr}_alpha{alpha}.txt", "a")
file1.write(str(all_value_means))
file1.write("\n") 
file1.close()

file1 = open(Path(f"Aggregators/{aggr}") / f"f1scoresW_{aggr}_alpha{alpha}.txt", "a")
file1.write(str(all_f1_w))
file1.write("\n") 
file1.close()"""